# Appendix: base models for weak supervision (Tables 4 and 5)

Confidence from the paired pre-trained (Base) model, with and without the
reasoning chain, compared to test-time self-consistency (TT-SC, 100 samples),
for the five Qwen3 models. Table 4 averages over datasets (by model), Table 5
over models (by dataset).

Inputs: `outputs/.../*_with_base_targets.csv` (`scripts/baselines/base_model_scores.py`)
and `results/main.csv` (TT-SC).

In [ ]:
import numpy as np
import pandas as pd

from basc import paths
from basc.metrics import correctness_metrics
from paper_utils import DATASETS_MAP, RESULTS_DIR, TABLES_DIR, promote_semantic

MODELS = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B-Thinking-2507", "Qwen/Qwen3-8B", "Qwen/Qwen3-14B"]
DATASETS = ["gsm8k", "polymath", "sciq", "trivia_qa", "webq"]
BASE_METHODS = ["base_response_avg_prob", "base_answer_avg_prob"]

In [ ]:
rows = []
for model in MODELS:
    for dataset in DATASETS:
        df = pd.read_csv(paths.base_model_scores(model, dataset))
        df = df[df["answer"].notna() & (df["answer"].astype(str).str.strip() != "")]
        for method in BASE_METHODS:
            rows.append({"model": model.split("/")[-1], "dataset": dataset, "method": method,
                         **correctness_metrics(df[method].to_numpy(dtype=float),
                                               df["correct"].to_numpy(dtype=int))})
base_df = pd.DataFrame(rows)

# TT-SC on the same models and datasets, seed-averaged (from the main sweep).
hdf = promote_semantic(pd.read_csv(RESULTS_DIR / "main.csv"))
hdf["model"] = hdf["model"].str.split("/").str[-1]
sc = (hdf[(hdf["method"] == "oracle_sc_k100") & hdf["model"].isin([m.split("/")[-1] for m in MODELS])
          & hdf["dataset"].isin(DATASETS)]
      .groupby(["model", "dataset"])[["ECE1", "ECE2", "MCE", "Brier", "AUROC"]].mean().reset_index())
sc["method"] = "Test-time SC"
comp_df = pd.concat([base_df, sc], ignore_index=True)

In [ ]:
SHOW = ["method", "ECE2", "Brier", "AUROC"]
LABELS = {"base_answer_avg_prob": "Base Answer Probs.", "base_response_avg_prob": "Base Response Probs.",
          "Test-time SC": "Test-time SC"}
ORDER = ["Base Answer Probs.", "Base Response Probs.", "Test-time SC"]

def summarize(by):
    out = comp_df[[by] + SHOW].groupby([by, "method"]).mean().reset_index()
    out["method"] = pd.Categorical(out["method"].map(LABELS), ORDER, ordered=True)
    if by == "dataset":
        out["dataset"] = out["dataset"].map(DATASETS_MAP)
    return out.sort_values([by, "method"]).reset_index(drop=True)

by_model = summarize("model")
tex_by_model = by_model.to_latex(index=False, float_format="%.3f")
(TABLES_DIR / "table4_base_models_by_model.tex").write_text(tex_by_model)
print(tex_by_model)
by_model

In [ ]:
by_dataset = summarize("dataset")
tex_by_dataset = by_dataset.to_latex(index=False, float_format="%.3f")
(TABLES_DIR / "table5_base_models_by_dataset.tex").write_text(tex_by_dataset)
print(tex_by_dataset)
by_dataset